In [4]:
import importlib.util, json, random, subprocess, sys
from pathlib import Path
_REQUIRED = {"pandas": "pandas", "pyarrow": "pyarrow", "networkx": "networkx"}
_missing = [package for module, package in _REQUIRED.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
import pandas as pd
import networkx as nx
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find shared config.json from the notebook working directory")
CONFIG = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))
SEED = int(CONFIG["random_seed"])
random.seed(SEED)
for key in ("data_dir", "artifact_dir", "results_dir"):
    (ROOT / CONFIG[key]).mkdir(parents=True, exist_ok=True)
ARTIFACTS = ROOT / CONFIG["artifact_dir"]
RESULTS = ROOT / CONFIG["results_dir"]
print(f"Setup complete; smoke_test_mode={CONFIG['smoke_test_mode']}; seed={SEED}")

Setup complete; smoke_test_mode=True; seed=42


Eligibility checks entry age (including term/PPT-specific rules), maturity age, requested policy term, and requested sum assured. Any known failure is `ineligible`; missing or unchecked required facts are `unknown`. The graph preserves LIC product and brochure links, verification scope, and catalogue status.

In [5]:
import re
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from tools.policy_eligibility import evaluate_eligibility

policies_df = pd.read_parquet(ARTIFACTS / "policies.parquet")
queries_df = pd.read_csv(ARTIFACTS / "queries.csv")

GOAL_KEYWORDS = {
    "Life Cover": ("term", "life cover", "protection", "death benefit", "sum assured"),
    "Child Education": ("child", "daughter", "son", "education", "school", "college", "marriage"),
    "Retirement": ("retirement", "pension", "annuity", "retire", "old age"),
    "Savings": ("savings", "endowment", "guaranteed", "money back", "bachat", "maturity"),
    "Wealth Creation": ("ulip", "wealth", "market", "equity", "high returns", "growth"),
    "Health Protection": ("cancer", "health", "hospital", "mediclaim", "illness", "medical", "icu"),
    "Income": ("regular income", "monthly income", "annual income", "survival benefit", "cash flow"),
}


def parse_intent(text):
    text = text if isinstance(text, str) else ""
    normalized = text.lower()
    child_matches = []
    for pattern in (
        r"(?P<age>\d{1,2})\s*(?:year|yr)s?\s*old\s*(?:daughter|son|child|kid|baby)\b",
        r"(?:daughter|son|child|kid)\s*(?:is|aged|of)\s*(?P<age>\d{1,2})\b",
    ):
        child_matches.extend(re.finditer(pattern, normalized))
    child_age = next((int(match.group("age")) for match in child_matches if int(match.group("age")) <= 25), None)

    customer_age = None
    customer_pattern = (
        r"\b(?:(?:i am|i'm|my age is|age|aged|at age|senior citizen aged|turning|turned)"
        r"\s*(?:a\s*)?(?P<explicit>\d{1,3})\b|"
        r"(?P<year_old>\d{1,3})\s*[- ]?\s*(?:years?|yrs?)\s*[- ]old\b)"
    )
    for match in re.finditer(customer_pattern, normalized):
        age_group = "explicit" if match.group("explicit") else "year_old"
        start, end = match.span(age_group)
        if any(start < child.end("age") and end > child.start("age") for child in child_matches):
            continue
        age = int(match.group(age_group))
        if 16 <= age <= 105:
            customer_age = age
            break

    term_match = re.search(
        r"\bterm\s*(?:of\s*)?(?P<term_first>\d{1,2})\s*(?:years?|yrs?)?\b|"
        r"\b(?:for\s+)?(?P<term_after>\d{1,2})\s*[- ]?\s*(?:years?|yrs?)\b"
        r"(?!\s*[- ]?old)(?:\s*(?:policy\s*)?term)?",
        normalized,
    )
    term = int(next(value for value in term_match.groupdict().values() if value is not None)) if term_match else None
    ppt_match = re.search(
        r"\b(?:ppt|premium[- ]paying term|premium payment term|paying term)\s*(?:of\s*)?(\d{1,2})\b",
        normalized,
    )
    premium_paying_term = int(ppt_match.group(1)) if ppt_match else None

    amount_match = re.search(
        r"(?<![\w.])(?:₹\s*|rs\.?\s*)?(?P<amount>\d[\d,]*(?:\.\d+)?)\s*"
        r"(?P<unit>crores?|cr|lakhs?|lacs?|l|k)\b",
        normalized,
    )
    sum_assured = None
    if amount_match:
        left = normalized[max(0, amount_match.start() - 48):amount_match.start()]
        right = normalized[amount_match.end():amount_match.end() + 32]
        amount_context = left + amount_match.group(0) + right
        explicit_cover = bool(re.search(r"\b(?:sum\s+(?:assured|insured)|cover(?:age)?)\b", amount_context))
        premium_frequency = bool(re.match(r"\s*(?:yearly|annually|per\s+year|a\s+month|monthly|per\s+month)\b", right))
        premium_prefix = bool(re.search(r"\b(?:premium|budget|under|below|up\s+to)(?:\s+of)?\s*$", left))
        if explicit_cover or not (premium_frequency or premium_prefix):
            amount = float(amount_match.group("amount").replace(",", ""))
            unit = amount_match.group("unit")
            multiplier = 10_000_000 if unit.startswith(("crore", "cr")) else 100_000 if unit.startswith(("lakh", "lac", "l")) else 1_000
            sum_assured = int(amount * multiplier)

    gender_match = re.search(r"\b(male|female|man|woman)\b", normalized)
    gender_map = {"man": "male", "woman": "female"}
    gender = gender_map.get(gender_match.group(1), gender_match.group(1)) if gender_match else None
    if gender is None and re.search(r"\b(?:my\s+wife|wife|she)\b", normalized):
        gender = "female"

    budget_match = re.search(
        r"\b(?:under|below|up\s+to|budget|premium)(?:\s+of)?\s*(?:₹\s*|rs\.?\s*)?"
        r"(?P<amount>\d[\d,]*(?:\.\d+)?)\s*(?P<unit>crores?|cr|lakhs?|lacs?|rupees?|rs\.?|k)?\b",
        normalized,
    )
    if budget_match is None:
        budget_match = re.search(
            r"(?P<amount>\d[\d,]*(?:\.\d+)?)\s*(?P<unit>crores?|cr|lakhs?|lacs?|rupees?|rs\.?|k)?"
            r"\s*(?:yearly|annually|per\s+year|a\s+month|monthly|per\s+month)\b",
            normalized,
        )
    budget = None if budget_match is None else {
        "amount": budget_match.group("amount").replace(",", ""),
        "unit": (budget_match.group("unit") or "rupees").lower(),
    }

    goals = [goal for goal, keywords in GOAL_KEYWORDS.items() if any(re.search(rf"\b{re.escape(keyword)}\b", normalized) for keyword in keywords)]
    risk = next((value for value, keywords in (("High", ("high risk", "equity", "aggressive", "market linked", "ulip")), ("Moderate", ("moderate risk", "balanced", "medium risk")), ("Low", ("safe", "low risk", "risk free", "guaranteed", "no risk", "zero risk", "secure"))) if any(keyword in normalized for keyword in keywords)), None)
    premium = next((value for value, keywords in (("Low", ("cheap", "low premium", "affordable", "budget", "small premium", "micro")), ("Moderate", ("moderate premium", "reasonable premium", "regular premium")), ("High", ("single premium", "lump sum", "high premium", "hni"))) if any(keyword in normalized for keyword in keywords)), None)
    contradictions = []
    if any(term in normalized for term in ("high returns", "market returns", "wealth")) and any(term in normalized for term in ("zero risk", "no risk", "risk free", "risk-free")):
        contradictions.append("High market returns conflict with a zero-risk requirement.")
    confidence = sum(value is not None for value in (customer_age, goals[0] if goals else None, risk, premium)) / 4
    return {
        "customer_age": customer_age,
        "customer_age_days": None,
        "child_age": child_age,
        "term": term,
        "premium_paying_term": premium_paying_term,
        "sum_assured": sum_assured,
        "gender": gender,
        "budget": budget,
        "all_goals": goals,
        "primary_goal": goals[0] if goals else None,
        "risk_tolerance": risk,
        "premium_level": premium,
        "contradictions": contradictions,
        "is_ambiguous": not any((customer_age is not None, term is not None, sum_assured is not None, goals, risk, premium, budget)),
        "confidence": round(confidence, 2),
    }

def eligibility(policy, intent):
    return evaluate_eligibility(policy, intent)

def recommend(query, policies=policies_df):
    intent = parse_intent(query)
    states = {row["policy_id"]: eligibility(row, intent) for row in policies.to_dict(orient="records")}
    eligible_ids = sorted(policy_id for policy_id, state in states.items() if state["status"] == "eligible")
    return {
        "status": "SUCCESS" if eligible_ids else "NO_ELIGIBLE_POLICIES",
        "query": query,
        "intent": intent,
        "eligibility": states,
        "recommendations": eligible_ids,
    }


def build_kg(policies):
    graph = nx.DiGraph()
    for row in policies.to_dict(orient="records"):
        pid = str(row["policy_id"])
        graph.add_node(
            pid,
            type="Policy",
            policy_name=str(row["policy_name"]),
            min_age="" if pd.isna(row["min_age"]) else int(row["min_age"]),
            max_age="" if pd.isna(row["max_age"]) else int(row["max_age"]),
            min_age_days="" if pd.isna(row["min_age_days"]) else int(row["min_age_days"]),
            min_entry_age_days="" if pd.isna(row["min_entry_age_days"]) else int(row["min_entry_age_days"]),
            min_maturity_age="" if pd.isna(row["min_maturity_age"]) else int(row["min_maturity_age"]),
            max_maturity_age="" if pd.isna(row["max_maturity_age"]) else int(row["max_maturity_age"]),
            min_term="" if pd.isna(row["min_term"]) else int(row["min_term"]),
            max_term="" if pd.isna(row["max_term"]) else int(row["max_term"]),
            min_sum_assured="" if pd.isna(row["min_sum_assured"]) else float(row["min_sum_assured"]),
            max_sum_assured="" if pd.isna(row["max_sum_assured"]) else float(row["max_sum_assured"]),
            max_sum_assured_unlimited=bool(row["max_sum_assured_unlimited"]),
            sum_assured_verified=bool(row["sum_assured_verified"]),
            rules_json=str(row["rules_json"]),
            policy_url=str(row["policy_url"]),
            brochure_url=str(row["brochure_url"]),
            statutory_status=str(row["statutory_status"]),
            status=str(row["status"]),
            verified=bool(row["verified"]),
            identity_verified=bool(row["identity_verified"]),
            age_verified=bool(row["age_verified"]),
            term_verified=bool(row["term_verified"]),
            category_verified=bool(row["category_verified"]),
            recommendation_pool=bool(row["recommendation_pool"]),
            supersedes=str(row["supersedes"]),
            withdrawn_versions=str(row["withdrawn_versions"]),
            verified_scope=str(row["verified_scope"]),
            brochure_source_note=str(row["brochure_source_note"]),
            claimed_verification=str(row["claimed_verification"]),
            source_url=str(row["source_url"]),
            source_note=str(row["source_note"]),
        )
        for attribute, value, relation in (
            ("Goal", row["financial_goal"], "HAS_GOAL"),
            ("Category", row["category"], "IN_CATEGORY"),
            ("Risk", row["risk_tolerance_inferred"], "SUITS_RISK"),
            ("Premium", row["premium_level_inferred"], "HAS_PREMIUM"),
        ):
            if pd.isna(value) or value == "" or (attribute == "Category" and not row["category_verified"]):
                continue
            node_id = f"{attribute}:{value}"
            graph.add_node(node_id, type=attribute, value=str(value))
            graph.add_edge(pid, node_id, relation=relation)
    return graph

kg = build_kg(policies_df)
nx.write_graphml(kg, ARTIFACTS / "kg.graphml")
intent_records = [{"query_id": row.query_id, "raw_query": row.raw_query, "intent": parse_intent(row.raw_query)} for row in queries_df.itertuples(index=False)]
(ARTIFACTS / "intents.json").write_text(json.dumps(intent_records, indent=2), encoding="utf-8")
print(f"Wrote KG with {kg.number_of_nodes()} nodes and {kg.number_of_edges()} edges; intents are query-derived and synthetic-only.")

Wrote KG with 73 nodes and 156 edges; intents are query-derived and synthetic-only.


## Assertions

In [6]:
# Exact boundaries and source-backed conditional constraints are enforced before ranking.
policy = {"status": "active", "recommendation_pool": True, "verified": True, "age_verified": True, "term_verified": True, "sum_assured_verified": True, "min_age": 18, "max_age": 65, "min_age_days": None, "min_entry_age_days": None, "min_term": 5, "max_term": 40, "max_maturity_age": 100, "max_sum_assured_unlimited": True, "source_note": "", "eligibility_conditions": "", "rules_json": ""}
for age in (18, 65):
    assert eligibility(policy, {"customer_age": age, "term": None})["status"] == "eligible"
outside_age = 17
assert eligibility(policy, {"customer_age": outside_age, "term": None})["status"] == "ineligible"
for term in (5, 40):
    assert eligibility(policy, {"customer_age": 18, "term": term})["status"] == "eligible"
assert eligibility(policy, {"customer_age": 18, "term": 41})["status"] == "ineligible"
assert eligibility(policy, {"customer_age": None, "term": None})["status"] == "unknown"
added_policy = policies_df.loc[policies_df["policy_id"] == "LIC_770"].iloc[0].to_dict()
assert eligibility(added_policy, {"customer_age": 40, "term": None})["status"] == "eligible"
assert eligibility(added_policy, {"customer_age": 40, "term": 20})["status"] == "eligible"
withdrawn = policies_df.loc[policies_df["policy_id"].eq("LIC_911")].iloc[0].to_dict()
assert eligibility(withdrawn, {"customer_age": 40, "term": 20})["status"] == "ineligible"
for policy_id in ("LIC_873", "LIC_867"):
    unresolved = policies_df.loc[policies_df["policy_id"].eq(policy_id)].iloc[0].to_dict()
    assert eligibility(unresolved, {"customer_age": 40, "term": 20})["status"] == "unknown"

bima_kavach = policies_df.loc[policies_df["policy_id"].eq("LIC_887")].iloc[0].to_dict()
assert eligibility(bima_kavach, {"customer_age": 40, "term": 20, "sum_assured": 5_000_000})["status"] == "ineligible"
jeevan_labh = policies_df.loc[policies_df["policy_id"].eq("LIC_736")].iloc[0].to_dict()
assert eligibility(jeevan_labh, {"customer_age": 59, "term": 16, "sum_assured": 500_000})["status"] == "eligible"
assert eligibility(jeevan_labh, {"customer_age": 40, "term": 20, "sum_assured": 500_000})["status"] == "ineligible"
assert eligibility(jeevan_labh, {"customer_age": 59, "term": 21, "sum_assured": 500_000})["status"] == "ineligible"
yuva_term = policies_df.loc[policies_df["policy_id"].eq("LIC_875")].iloc[0].to_dict()
assert eligibility(yuva_term, {"customer_age": 45, "term": 40})["status"] == "ineligible"

parent = parse_intent("I am 36 with a 3 year old son")
assert (parent["customer_age"], parent["child_age"]) == (36, 3)
assert parse_intent("5 Lakhs")["customer_age"] is None
term_only = parse_intent("I need a 30 years term")
assert term_only["customer_age"] is None and term_only["term"] == 30
parsed = parse_intent("I need ₹50 lakh cover for a 20-year term at age 40")
assert (parsed["sum_assured"], parsed["term"], parsed["customer_age"]) == (5_000_000, 20, 40)
assert parse_intent("2 crore sum assured")["sum_assured"] == 20_000_000
assert parse_intent("45-year-old wants cover") ["customer_age"] == 45
assert parse_intent("I want high market returns with zero risk")["contradictions"]
empty = recommend("")
assert empty["intent"]["confidence"] == 0.0
assert empty["status"] == "NO_ELIGIBLE_POLICIES" and not empty["recommendations"]
old = recommend("I am 95 years old and want a policy")
assert old["status"] == "NO_ELIGIBLE_POLICIES" and not old["recommendations"]
assert all(state["status"] != "eligible" for state in old["eligibility"].values())
assert recommend("I am 36 years old looking for life cover")["recommendations"] == recommend("I am 36 years old looking for life cover")["recommendations"]
loaded_graph = nx.read_graphml(ARTIFACTS / "kg.graphml")
assert sum(node_data.get("type") == "Policy" for _, node_data in loaded_graph.nodes(data=True)) == len(policies_df)
assert loaded_graph.nodes["LIC_859"]["identity_verified"] in (True, "True")
assert loaded_graph.nodes["LIC_859"]["verified"] in (True, "True")
assert loaded_graph.nodes["LIC_911"]["status"] == "withdrawn"
assert loaded_graph.nodes["LIC_736"]["brochure_url"].startswith("https://licindia.in/")
print("Notebook 02 checks passed: intent parsing and brochure-backed age, maturity, term, and sum-assured gates.")

Notebook 02 checks passed: age/term gates, unknown status, parser, strict refusal, determinism, and GraphML provenance.


In [ ]:
human_eval_queries = pd.read_csv(ARTIFACTS / "human_eval_queries.csv", dtype=str).fillna("")
human_eval_intents = [
    {"query_id": row.query_id, "raw_query": row.raw_query, "intent": parse_intent(row.raw_query)}
    for row in human_eval_queries.itertuples(index=False)
]
if set(row["query_id"] for row in human_eval_intents) & set(queries_df["query_id"]):
    raise ValueError("Human-evaluation query IDs must be separate from synthetic smoke queries")
(ARTIFACTS / "human_eval_intents.json").write_text(json.dumps(human_eval_intents, indent=2), encoding="utf-8")
assert len(human_eval_intents) == len(human_eval_queries)
print(f"Parsed {len(human_eval_intents)} separate human-evaluation queries.")

parser_regression_cases = [
    {
        "case_id": "parser_case_1",
        "raw_query": "I'm 30 and have a small budget, want a basic policy with 5 lakh coverage for 10 year term",
        "corrected_intent": {"customer_age": 30, "term": 10, "sum_assured": 500000, "gender": None, "budget": None, "all_goals": []},
    },
    {
        "case_id": "parser_case_2",
        "raw_query": "Daily wage worker aged 35, want small premium policy with 5 lakh sum assured, 15 year term",
        "corrected_intent": {"customer_age": 35, "term": 15, "sum_assured": 500000, "gender": None, "budget": None, "all_goals": []},
    },
    {
        "case_id": "parser_case_3",
        "raw_query": "Age 26, first time buyer, need low premium cover of 10 lakh for 20 years term",
        "corrected_intent": {"customer_age": 26, "term": 20, "sum_assured": 1000000, "gender": None, "budget": None, "all_goals": []},
    },
    {
        "case_id": "parser_case_4",
        "raw_query": "Looking for 50 lakh term cover at age 40 for 20 years",
        "corrected_intent": {"customer_age": 40, "term": 20, "sum_assured": 5000000, "gender": None, "budget": None, "all_goals": []},
    },
    {
        "case_id": "parser_case_5",
        "raw_query": "I'm 37, want to put 1 lakh yearly in an index linked plan",
        "corrected_intent": {"customer_age": 37, "term": None, "sum_assured": None, "gender": None, "budget": {"amount": "1", "unit": "lakh"}, "all_goals": []},
    },
]
corrected_intent_audit_inputs = [
    {**case, "parsed_intent": parse_intent(case["raw_query"])}
    for case in parser_regression_cases
]
(ARTIFACTS / "corrected_intent_audit_inputs.json").write_text(
    json.dumps(corrected_intent_audit_inputs, indent=2), encoding="utf-8"
)
assert len(corrected_intent_audit_inputs) == 5

